In [1]:
# Verbindung zu Google Drive aufbauen
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:

# NOTEBOOK 1 – PREPROCESSING
#  Aufgabe: Bänder clippen + Wolkenmaske berechnen & speichern
#  Output:  Landsat/Clipped/ → geclippte Bänder + CLEAR_MASK pro Szene


import rasterio
import numpy as np
import geopandas as gpd
import os
from rasterio.mask import mask as rio_mask

# SETUP

base      = "/content/drive/MyDrive/Colab_Notebooks"
raw_path  = f"{base}/Landsat/raw"
clip_path = f"{base}/Landsat/Clipped"
alkis_path = f"{base}/ALKIS/ALKIS_Stadt_Wue/ALKIS_Stadt_Wue.shp"

# Alle Ausgabeordner anlegen
for ordner in [
    f"{base}/Landsat/Clipped",
    f"{base}/Landsat/Processed",
    f"{base}/NDVI",
    f"{base}/NDVI/Szenen",
    f"{base}/LST",
    f"{base}/LST/Szenen",
    f"{base}/Albedo",
    f"{base}/Albedo/Szenen",
    f"{base}/Grafiken",
]:
    os.makedirs(ordner, exist_ok=True)

print("Ordnerstruktur angelegt")

# ALKIS laden
alkis = gpd.read_file(alkis_path)
print(f"ALKIS geladen | CRS: {alkis.crs}")


# Szenen-IDs aus QA_PIXEL Dateien ableiten
scene_ids = sorted(set(
    f.replace("_QA_PIXEL.TIF", "")
    for f in os.listdir(raw_path)
    if f.endswith("_QA_PIXEL.TIF")
))

print(f"\n {len(scene_ids)} Szenen gefunden:")
for s in scene_ids:
    print(f"  - {s}")

# Welche Bänder clippen:
BENOETIGTE_BAENDER = [
    "QA_PIXEL",  # Wolkenmaske
    "SR_B2",     # Albedo (Blau)
    "SR_B4",     # NDVI + Albedo (Rot)
    "SR_B5",     # NDVI + Albedo (NIR)
    "SR_B6",     # Albedo (SWIR1)
    "SR_B7",     # Albedo (SWIR2)
    "ST_B10",    # LST (thermisch)
]

#Wolkenmaske berechnen & speichern

def extract_clear_mask(qa_array):
    """
    Erstellt binäre Wolkenmaske aus QA_Pixel-Band.
    Bits: 1=aufgeweitete Wolke, 3=Wolke, 4=Wolkenschatten, 5=Schnee
    Rückgabe: 1 = klarer Pixel, 0 = Wolke/Schatten/Schnee
    """
    dilated_cloud = (qa_array & (1 << 1)) > 0
    cloud         = (qa_array & (1 << 3)) > 0
    cloud_shadow  = (qa_array & (1 << 4)) > 0
    snow          = (qa_array & (1 << 5)) > 0
    return (~(dilated_cloud | cloud | cloud_shadow | snow)).astype(np.uint8)


for scene_id in scene_ids:
    print(f"\n{'='*60}")
    print(f"🔄 {scene_id}")

    # Bänder clippen
    clip_meta_ref = None  # Metadaten für Wolkenmaske

    for band in BENOETIGTE_BAENDER:
        in_path  = os.path.join(raw_path,  f"{scene_id}_{band}.TIF")
        out_path = os.path.join(clip_path, f"{scene_id}_{band}_clip.tif")

        # Prüfen ob Quelldatei vorhanden
        if not os.path.exists(in_path):
            print(f"  ⚠️ {band} nicht gefunden – übersprungen!")
            continue

        # Prüfen ob bereits geclippt (nicht nochmal verarbeiten)
        if os.path.exists(out_path):
            print(f"  ⏭ {band} bereits geclippt – übersprungen")
            if clip_meta_ref is None and band == "QA_PIXEL":
                with rasterio.open(out_path) as src:
                    clip_meta_ref = src.meta.copy()
            continue

        with rasterio.open(in_path) as src:
            alkis_reproj = alkis.to_crs(src.crs)
            geoms        = list(alkis_reproj.geometry)
            clipped, clip_transform = rio_mask(src, geoms, crop=True)
            clip_meta = src.meta.copy()
            clip_meta.update({
                "height":    clipped.shape[1],
                "width":     clipped.shape[2],
                "transform": clip_transform,
                "driver":    "GTiff"
            })

            # QA_PIXEL Metadaten für Wolkenmaske merken
            if band == "QA_PIXEL":
                clip_meta_ref = clip_meta.copy()

        with rasterio.open(out_path, "w", **clip_meta) as dst:
            dst.write(clipped)

        print(f"  ✔ {band} geclippt")

    # ── Schritt 3b: Wolkenmaske berechnen & speichern ─────────────────────────
    mask_path = os.path.join(clip_path, f"{scene_id}_CLEAR_MASK.tif")

    if os.path.exists(mask_path):
        print(f"  ⏭ CLEAR_MASK bereits vorhanden – übersprungen")
        # Statistik trotzdem ausgeben
        with rasterio.open(mask_path) as src:
            clear_mask = src.read(1)
        pct = clear_mask.sum() / clear_mask.size * 100
        print(f"  ✔ Klare Pixel: {clear_mask.sum():,} / "
              f"{clear_mask.size:,} ({pct:.1f}%)")
        continue

    # QA_PIXEL geclippt laden
    qa_clip_path = os.path.join(clip_path, f"{scene_id}_QA_PIXEL_clip.tif")
    if not os.path.exists(qa_clip_path):
        print(f"  ❌ QA_PIXEL_clip fehlt – Wolkenmaske nicht berechnet!")
        continue

    with rasterio.open(qa_clip_path) as src:
        qa = src.read(1)
        qa_meta = src.meta.copy()

    # Wolkenmaske berechnen
    clear_mask = extract_clear_mask(qa)
    pct = clear_mask.sum() / clear_mask.size * 100
    print(f"  ✔ Klare Pixel: {clear_mask.sum():,} / "
          f"{clear_mask.size:,} ({pct:.1f}%)")

    # ✅ mask_meta mit korrekter Einrückung (innerhalb der for-Schleife!)
    mask_meta = {
        "driver":    "GTiff",
        "dtype":     rasterio.uint8,
        "count":     1,
        "nodata":    255,
        "height":    qa_meta["height"],
        "width":     qa_meta["width"],
        "transform": qa_meta["transform"],
        "crs":       qa_meta["crs"],
    }

    # Diagnose
    print("dtype:  ", mask_meta["dtype"])
    print("nodata: ", mask_meta["nodata"])

    # Speichern
    with rasterio.open(mask_path, "w", **mask_meta) as dst:
        dst.write(clear_mask, 1)
    print(f"  ✔ CLEAR_MASK gespeichert")

# SCHRITT 4: Überprüfung – alle Dateien vorhanden?


print(f"\n{'='*60}")
print("📋 ÜBERPRÜFUNG – Clipped Ordner:")
print(f"{'='*60}")

# Erwartete Dateien pro Szene
erwartete_suffixe = [f"_{b}_clip.tif" for b in BENOETIGTE_BAENDER]
erwartete_suffixe.append("_CLEAR_MASK.tif")

alle_ok = True
for scene_id in scene_ids:
    fehlend = []
    for suffix in erwartete_suffixe:
        pfad = os.path.join(clip_path, f"{scene_id}{suffix}")
        if not os.path.exists(pfad):
            fehlend.append(suffix)
    if fehlend:
        print(f"  ⚠️ {scene_id}: fehlend → {fehlend}")
        alle_ok = False
    else:
        print(f"  ✅ {scene_id}: alle {len(erwartete_suffixe)} Dateien vorhanden")

if alle_ok:
    print(f"\n✅ Preprocessing abgeschlossen!")
    print(f"   {len(scene_ids)} Szenen × {len(erwartete_suffixe)} Dateien "
          f"= {len(scene_ids) * len(erwartete_suffixe)} Dateien in Clipped/")
else:
    print(f"\n⚠️ Einige Dateien fehlen – bitte prüfen!")

Ordnerstruktur angelegt
ALKIS geladen | CRS: EPSG:25832

 5 Szenen gefunden:
  - LC08_L2SP_194025_20250701_20250711_02_T1
  - LC08_L2SP_194025_20250818_20250821_02_T1
  - LC08_L2SP_194025_20250919_20250929_02_T1
  - LC09_L2SP_194025_20250810_20250811_02_T1
  - LC09_L2SP_194025_20250826_20250902_02_T1

🔄 LC08_L2SP_194025_20250701_20250711_02_T1
  ⏭ QA_PIXEL bereits geclippt – übersprungen
  ⏭ SR_B2 bereits geclippt – übersprungen
  ⏭ SR_B4 bereits geclippt – übersprungen
  ⏭ SR_B5 bereits geclippt – übersprungen
  ⏭ SR_B6 bereits geclippt – übersprungen
  ⏭ SR_B7 bereits geclippt – übersprungen
  ⏭ ST_B10 bereits geclippt – übersprungen
  ⏭ CLEAR_MASK bereits vorhanden – übersprungen
  ✔ Klare Pixel: 148,895 / 171,500 (86.8%)

🔄 LC08_L2SP_194025_20250818_20250821_02_T1
  ⏭ QA_PIXEL bereits geclippt – übersprungen
  ⏭ SR_B2 bereits geclippt – übersprungen
  ⏭ SR_B4 bereits geclippt – übersprungen
  ⏭ SR_B5 bereits geclippt – übersprungen
  ⏭ SR_B6 bereits geclippt – übersprungen
  ⏭ SR_B

In [4]:

# SCHRITT 5: Gebiete laden & Hubland-Gesamt erstellen
import pandas as pd

shapefile_pfade = {
    "Rottenbauer-Nord":     f"{base}/Shapefiles_Gebiete/Rottenbauer/Shape_Rottenbauer_Nord.shp",
    "Rottenbauer-Süd":      f"{base}/Shapefiles_Gebiete/Rottenbauer/Shape_Rottenbauer_alt.shp",
    "Lengfeld":             f"{base}/Shapefiles_Gebiete/Lengfeld/Shape_Lengfeld.shp",
    "Lengfeld-Kern":        f"{base}/Shapefiles_Gebiete/Lengfeld/Shape_Lengfeld_alt.shp",
    "Hubland Grünfläche":   f"{base}/Shapefiles_Gebiete/Hubland/Shape_Hubland_Gruenflaeche.shp",
    "Hubland Terrassen":    f"{base}/Shapefiles_Gebiete/Hubland/Shape_Hublandterrassen.shp",
    "Hubland Quartier I":   f"{base}/Shapefiles_Gebiete/Hubland/Shape_Quartier_I.shp",
    "Hubland Quartier II":  f"{base}/Shapefiles_Gebiete/Hubland/Shape_Quartier_II_plus_BBP_70.shp",
    "Hubland Quartier III": f"{base}/Shapefiles_Gebiete/Hubland/Shape_Quartier_III.shp",
    "Gartenstadt":          f"{base}/Shapefiles_Gebiete/Gartenstadt/Shape_Gartenstadt.shp",
}

# Alle Shapefiles laden
gebiete_gdfs = {name: gpd.read_file(pfad)
                for name, pfad in shapefile_pfade.items()}

# Hubland-Gesamt zusammenführen
hubland_gesamt = gpd.GeoDataFrame(
    geometry=[pd.concat([
        gebiete_gdfs["Hubland Grünfläche"].geometry,
        gebiete_gdfs["Hubland Terrassen"].geometry,
        gebiete_gdfs["Hubland Quartier I"].geometry,
        gebiete_gdfs["Hubland Quartier II"].geometry,
        gebiete_gdfs["Hubland Quartier III"].geometry,
    ]).union_all()],
    crs=gebiete_gdfs["Hubland Grünfläche"].crs
)

# Hubland-Gesamt als Shapefile speichern
hubland_path = f"{base}/Shapefiles_Gebiete/Hubland/Shape_Hubland_Gesamt.shp"
hubland_gesamt.to_file(hubland_path)
print(f"✅ Hubland-Gesamt gespeichert: {hubland_path}")

# Gebiete final zusammenstellen
gebiete_final = {}
for name, gdf in gebiete_gdfs.items():
    gebiete_final[name] = gdf
    if name == "Hubland Quartier III":
        gebiete_final["Hubland-Gesamt"] = hubland_gesamt

print(f"✅ {len(gebiete_final)} Gebiete bereit:")
for name in gebiete_final.keys():
    print(f"  - {name}")

✅ Hubland-Gesamt gespeichert: /content/drive/MyDrive/Colab_Notebooks/Shapefiles_Gebiete/Hubland/Shape_Hubland_Gesamt.shp
✅ 11 Gebiete bereit:
  - Rottenbauer-Nord
  - Rottenbauer-Süd
  - Lengfeld
  - Lengfeld-Kern
  - Hubland Grünfläche
  - Hubland Terrassen
  - Hubland Quartier I
  - Hubland Quartier II
  - Hubland Quartier III
  - Hubland-Gesamt
  - Gartenstadt
